In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import sys
sys.path.insert(0, "/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/diffusers/src")
from diffusers.replica_exchange.acceptance import _k_ladder

from diffusers import StableDiffusion3Pipeline
import numpy as np

In [3]:
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir="/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/model_checkpoints"
)
pipe = pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [4]:
swap_algorithm={
		"n_replicas": 3, 
		"p_ratio": "p",
		"even_indices": [0, 7, 12, 16, 19, 21],   # t ≈ 870, 763, 648, 536
		"odd_indices":  [1, 8, 13, 17, 20, 22],
		"debug": True
	}

tsr_sigma = 3.0
labels = ["chocolates", "badminton", "princess", "buildings", "canoe"]
prompts = ["a box of chocolates", "Boy playing badminton with his grandfather", "The Princess and the Frog Read-Along W/CD [With Paperback Book]", "London from the Sky Garden Photographic Print", "Canoe on Elk Lake"]

In [5]:
tsr_k_vals = [1.4, 0.6]
replica_exchanges = [True, False]

for i, prompt in enumerate(prompts):

	for tsr_k in tsr_k_vals:
		for replica_exchange in replica_exchanges:

			generator = torch.Generator(device="cuda").manual_seed(42)

			all_images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=30,
				guidance_scale=5.0,
				tsr_k=tsr_k,
				tsr_sigma=tsr_sigma,
				replica_exchange=replica_exchange,
				swap_algorithm=swap_algorithm,
				generator=generator,
			).images
			

			for idx in range(len(all_images)):
				image = all_images[idx]
				arr = np.array(image)
				print(f"Image {idx}: min={arr.min()}, max={arr.max()}, mean={arr.mean():.2f}")

				if replica_exchange:
					k_ladder = _k_ladder(torch.tensor(tsr_k), swap_algorithm["n_replicas"], device="cpu", dtype=torch.float32)				
					k_val = k_ladder[idx]
					string = f"pt_tsr_{tsr_k}_val_{k_val:.2f}"
				else:
					string = f"tsr_{tsr_k}"

				output_path = f"/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/{labels[i]}_{string}.png"
				image.save(output_path)
				print(f"Saved to {output_path}")

We will be running with replica swaps with 3 replicas


  0%|          | 0/30 [00:00<?, ?it/s]

 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 1000.0
Time 1000.0 swap btwn source 1.00 and target 1.40 accept 1.000 std 0.986
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 988.2713623046875
Time 988.2713623046875 swap btwn source 0.71 and target 1.00 accept 1.000 std 0.975
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 975.9791870117188
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 963.08203125
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 949.5339965820312
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 935.2844848632812
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 920.2777099609375
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 904.4515380859375
Time 904.4515380859375 swap btwn source 1.00 and target 1.40 accept 1.000 std 0.903
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 887.737060546875
Time 887.737060546875 swap btwn source 0.71 and target 1.00 accept 1.000 std 0.890
 We tsr by 1.40
 We tsr by 1.00
 We tsr by 0.71
t is 8

OutOfMemoryError: CUDA out of memory. Tried to allocate 512.00 MiB. GPU 0 has a total capacity of 19.62 GiB of which 295.31 MiB is free. Process 94482 has 880.00 MiB memory in use. Process 98948 has 14.96 GiB memory in use. Including non-PyTorch memory, this process has 18.36 GiB memory in use. Of the allocated memory 17.29 GiB is allocated by PyTorch, and 870.23 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)